# 05 Forecast and inventory

Lessons 10-11: how much to prep, what Sep-Nov 2026 will sell, and profit by scenario.

Data: daily panel (kitchen x item x day) from `orders_enriched.parquet` + filled `waste_daily`.
Assumption: prepared = sold + wasted (no prep log in the data). No grams per cup either, so plans are in cups and THB.

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
from IPython.display import display
from scipy.stats import norm

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))  # lets the notebook import src/

from src import scenario as sc
from src.forecast import (DAY_KEYS, critical_ratio, daily_panel, demand_cv, forecast_all,
                          forecast_horizon, normal_waste_short, prep_rule, score_prep, wape_bias)
from src.load_clean import BERRY, load_tables

# Same chart setup as notebooks 02-04.
pio.renderers.default = "plotly_mimetype+notebook_connected"
pio.templates["fb"] = go.layout.Template(layout=dict(margin=dict(t=118, l=70, r=30, b=60), font_size=13))
pio.templates.default = "plotly_white+fb"
OUT = ROOT / "outputs"

ITEMS = ["Watermelon", "Pineapple", "Guava", "PassionFruit", "MixedBerryPremium"]
ITEM_COLORS = dict(zip(ITEMS, ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4"]))
UP, DOWN, TOTAL, INK = "#2a78d6", "#e34948", "#52514e", "#0b0b0b"
COST_MONTHS = ["2026-06", "2026-07", "2026-08"]   # latest costs; also the rule backtest window
STEADY = "2026-08"                                # rule window no longer holds the April peak


def subtitle(title, sub):
    """Chart title: the finding (split at ';' to fit), then a grey line saying what is plotted."""
    sub_html = f"<span style='font-size:12px;color:#52514e'>{sub}</span>"
    return dict(text=title.replace("; ", ";<br>") + "<br>" + sub_html, font_size=16,
                x=0.01, xanchor="left", y=1, yref="container", yanchor="top", pad=dict(t=36))


sheets = load_tables()
e = pd.read_parquet(ROOT / "data" / "processed" / "orders_enriched.parquet")
p = daily_panel(e, sheets)
promo_inc = pd.read_csv(OUT / "promo_incremental.csv")
LIFT = sc.promo_lift(promo_inc)   # lesson 7: cups on promo / baseline cups
assert p["sold"].sum() == e["units_sold"].sum()
assert np.isclose(p["waste_cost"].sum(), sheets["waste_daily"]["est_waste_cost_thb"].sum())
print(f"{len(p):,} kitchen-item-days | {p['sold'].sum():,.0f} cups sold | "
      f"{p['wasted'].sum():,.0f} cups wasted | waste {p['waste_cost'].sum():,.0f} THB")

6,576 kitchen-item-days | 618,123 cups sold | 36,680 cups wasted | waste 860,307 THB


## Part A: waste and the prep rule

## 1. Where is the waste?

In [2]:
def waste_table(df, by):
    """Waste cups, rate and THB a month by group. Prepared = sold + wasted."""
    g = df.groupby(by).agg(sold=("sold", "sum"), wasted=("wasted", "sum"), waste_cost=("waste_cost", "sum"),
                           months=("month", "nunique"), kitchen_days=("sold", "size"))
    g["waste_rate_pct"] = g["wasted"] / (g["sold"] + g["wasted"]) * 100
    g["thb_per_month"] = g["waste_cost"] / g["months"]
    g["sold_per_kitchen_day"] = g["sold"] / g["kitchen_days"]
    g["wasted_per_kitchen_day"] = g["wasted"] / g["kitchen_days"]
    return g


SHOW = ["waste_rate_pct", "thb_per_month", "sold_per_kitchen_day", "wasted_per_kitchen_day"]
by_item = waste_table(p, "sku").reindex(ITEMS)
by_kitchen = waste_table(p, "kitchen")
by_dow = waste_table(p, "dow")
by_promo = waste_table(p, ["sku", "promo_day"])
display(by_item[SHOW].round(1))
display(by_kitchen[SHOW].round(1))
display(by_dow[SHOW].round(1))
display(by_promo[SHOW].round(1))

zero = (p["wasted"] == 0).groupby(p["sku"]).mean().reindex(ITEMS) * 100
print("Zero-waste days, % of kitchen-item-days:", zero.round(2).to_dict())
berry_share = by_item.loc[BERRY, "waste_cost"] / by_item["waste_cost"].sum() * 100
berry_cups = by_item.loc[BERRY, "sold"] / by_item["sold"].sum() * 100
print(f"Berry: {berry_share:.0f}% of waste THB with {berry_cups:.1f}% of cups sold")

frames = []
for level, t in [("item", by_item), ("kitchen", by_kitchen), ("weekday", by_dow), ("item x promo day", by_promo)]:
    t = t.reset_index()
    t["group"] = t.drop(columns=t.columns[t.columns.get_loc("sold"):]).astype(str).agg(" | ".join, axis=1)
    frames.append(t.assign(level=level)[["level", "group", "sold", "wasted", "waste_cost", "waste_rate_pct",
                                         "thb_per_month", "sold_per_kitchen_day", "wasted_per_kitchen_day"]])
pd.concat(frames).to_csv(OUT / "waste_summary.csv", index=False)

,waste_rate_pct,thb_per_month,sold_per_kitchen_day,wasted_per_kitchen_day
sku,,,,
Watermelon,3.3,9478.8,167.8,5.7
Pineapple,4.4,9536.9,111.1,5.1
Guava,5.5,7466.9,69.5,4.0
PassionFruit,8.8,17298.3,55.8,5.4
MixedBerryPremium,20.5,55822.8,38.1,9.9


,waste_rate_pct,thb_per_month,sold_per_kitchen_day,wasted_per_kitchen_day
kitchen,,,,
BKK_Ladprao,8.3,16377.9,55.6,5.0
BKK_Sukhumvit,5.1,16116.8,93.0,5.0
Pattaya_Beach,5.1,18034.9,104.3,5.6
Pattaya_Central,5.1,21162.6,123.1,6.7


,waste_rate_pct,thb_per_month,sold_per_kitchen_day,wasted_per_kitchen_day
dow,,,,
0,5.7,9960.2,89.4,5.4
1,5.6,10412.4,94.8,5.6
2,5.6,9692.9,89.8,5.3
3,5.6,9725.3,90.5,5.4
4,5.6,9715.8,89.8,5.3
5,5.5,10887.4,102.7,6.0
6,5.7,11298.2,101.0,6.1


waste_rate_pct  thb_per_month  \
sku               promo_day                                  
Guava             False                 5.5         6657.5   
                  True                  5.3          809.4   
MixedBerryPremium False                20.5        47776.2   
                  True                 21.0         8046.6   
PassionFruit      False                 8.8        15356.5   
                  True                  8.8         1941.7   
Pineapple         False                 4.4         8619.0   
                  True                  4.3          917.9   
Watermelon        False                 3.3         8361.1   
                  True                  3.3         1117.8   

                             sold_per_kitchen_day  wasted_per_kitchen_day  
sku               promo_day                                                
Guava             False                      67.6                     3.9  
                  True                       90.1                     5.1  
MixedBerryPremium False                      37.3                     9.6  
                  True                       43.8                    11.7  
PassionFruit      False                      54.4                     5.2  
                  True                       70.8                     6.8  
Pineapple         False                     108.5                     5.0  
                  True                      141.7                     6.3  
Watermelon        False                     163.0                     5.5  
                  True                      216.4                     7.4

Zero-waste days, % of kitchen-item-days: {'Watermelon': 0.07, 'Pineapple': 0.14, 'Guava': 0.07, 'PassionFruit': 0.07, 'MixedBerryPremium': 0.27}
Berry: 39% of waste THB with 4.5% of cups sold


In [3]:
fig = go.Figure(go.Bar(
    y=by_item.index, x=by_item["thb_per_month"] / 1000, orientation="h",
    marker_color=[ITEM_COLORS[s] for s in by_item.index],
    text=[f"{v / 1000:.1f}k ({r:.1f}%)" for v, r in zip(by_item["thb_per_month"], by_item["waste_rate_pct"])],
    textposition="outside", hovertemplate="%{y}<br>%{x:.1f}k THB a month<extra></extra>"))
fig.update_layout(
    title=subtitle(f"Berry is {berry_share:.0f}% of waste THB with {berry_cups:.0f}% of cups; "
                   "the 3 cheap items waste 3-6%",
                   "Waste cost THB thousands a month (months on sale), label = waste rate (wasted / prepared)"),
    xaxis=dict(title="THB thousands a month", range=[0, by_item["thb_per_month"].max() / 1000 * 1.3]),
    yaxis=dict(autorange="reversed", title=None), showlegend=False, height=420)
fig.show()

**Berry: 39% of waste THB (55.8k a month) with 4.5% of cups, waste rate 20.5%. PassionFruit next: 8.8%, 17.3k a month.** The 3 cheap items waste 3.3-5.5%.
Weekdays: flat, 5.5-5.7%. Kitchens: Ladprao 8.3% vs 5.1% at the other 3 (lesson 9).
Promo days: same waste rate as normal days for every item; they sell ~30% more and waste ~30% more cups. So kitchens already prep more on promo days.
Zero-waste days: 0.07-0.27% of kitchen-item-days, so a sold-out almost never shows. Limit: sales = what was sold, not what customers wanted (censored demand).

## 2. How many cups should a kitchen prep? (critical ratio)

In [4]:
ja = p[p["month"].isin(COST_MONTHS)]
cr = ja.groupby("sku")[["sold", "cm", "wasted", "waste_cost"]].sum().reindex(ITEMS)
cr["cu"] = cr["cm"] / cr["sold"]              # profit lost when a customer cannot buy
cr["co"] = cr["waste_cost"] / cr["wasted"]    # money lost on a cup we throw away
cr["cr"] = critical_ratio(cr["cu"], cr["co"])
cr["z"] = norm.ppf(cr["cr"])
display(cr[["cu", "co", "cr", "z"]].round(2))

,cu,co,cr,z
sku,,,,
Watermelon,13.00,13.95,0.48,-0.04
Pineapple,14.45,15.23,0.49,-0.03
Guava,12.98,16.24,0.44,-0.14
PassionFruit,6.26,33.68,0.16,-1.01
MixedBerryPremium,2.56,53.07,0.05,-1.64


**Cheap items: CR 0.44-0.49 (z about 0), so prep about a median day. PassionFruit 0.16, berry 0.05: prep well below the median.**
Cu = CM per cup Jun-Aug (profit lost when a customer cannot buy). Co = waste cost per wasted cup (fruit + packaging).
A wasted berry costs 53 THB, a lost berry sale only 2.6 THB (rainy-season fruit cost), so running short is cheap and waste is expensive.

## 3. Does the rule beat today's prep? (backtest Jun-Aug 2026)

In [5]:
p["today_cost"] = p["wasted"] * p["sku"].map(cr["co"])


def backtest_rule(q, months, lift=None):
    """Prep rule on the given months; waste, short and cost per kitchen-item-day."""
    b = p[p["month"].isin(months)].join(prep_rule(p, q, lift=lift))
    assert b["prep"].notna().all(), "rule has no history for some days"
    return b.join(score_prep(b["sold"], b["prep"], b["sku"].map(cr["cu"]), b["sku"].map(cr["co"])))


def compare(b, by):
    """Today vs rule: cups, short rate and cost saving a month."""
    g = b.groupby(by).agg(sold=("sold", "sum"), today_waste=("wasted", "sum"), rule_waste=("waste", "sum"),
                          rule_short=("short", "sum"), today_cost=("today_cost", "sum"),
                          rule_cost=("cost", "sum"), months=("month", "nunique"))
    g["short_rate_pct"] = g["rule_short"] / g["sold"] * 100
    g["saving_per_month"] = (g["today_cost"] - g["rule_cost"]) / g["months"]
    return g.drop(columns="months")


bt = backtest_rule(cr["cr"].to_dict(), COST_MONTHS)
print("Jun-Aug 2026, rule at CR")
display(compare(bt, "sku").reindex(ITEMS).round(1))
display(compare(bt, "month").round(1))

Jun-Aug 2026, rule at CR


,sold,today_waste,rule_waste,rule_short,today_cost,rule_cost,short_rate_pct,saving_per_month
sku,,,,,,,,
Watermelon,51597,1765.0,10108.0,1613.0,24623.3,161990.6,3.1,-45789.1
Pineapple,33874,1515.0,6852.0,1112.0,23070.8,120416.0,3.3,-32448.4
Guava,21188,1209.0,4249.0,1175.0,19632.2,84245.2,5.5,-21537.6
PassionFruit,17048,1691.0,1517.0,1794.0,56955.2,62331.3,10.5,-1792.0
MixedBerryPremium,8573,2155.0,853.0,1187.0,114367.2,48312.3,13.8,22018.3


,sold,today_waste,rule_waste,rule_short,today_cost,rule_cost,short_rate_pct,saving_per_month
month,,,,,,,,
2026-06,45183,2977.0,15282.0,898.0,87171.7,274448.8,2.0,-187277.1
2026-07,43292,2673.0,6360.0,2068.0,76829.6,127393.3,4.8,-50563.7
2026-08,43805,2685.0,1937.0,3915.0,74647.5,75453.3,8.9,-805.9


In [6]:
fig = go.Figure()
by_m = compare(bt, "month")
for col, name, color in [("today_waste", "Today: wasted", TOTAL), ("rule_waste", "Rule: wasted", UP),
                         ("rule_short", "Rule: short", DOWN)]:
    fig.add_bar(x=by_m.index, y=by_m[col], name=name, marker_color=color,
                hovertemplate="%{x}<br>" + name + ": %{y:,.0f} cups<extra></extra>")
fig.update_layout(
    barmode="group",
    title=subtitle("The rule needs 8 weeks to forget the April peak; by August it wastes less "
                   "but runs short on more cups",
                   "Cups a month, all kitchens and items. Rule = CR quantile of the last 8 normal same-weekday days"),
    yaxis_title="Cups", height=440, legend=dict(orientation="h", y=1.02, x=1, xanchor="right"))
fig.show()

In [7]:
rows, runs = [], {}
for label, add, lift in [("CR", 0.0, None), ("CR+0.1", 0.1, None), ("CR+0.1, promo x lift", 0.1, LIFT)]:
    q = {s: min(cr.loc[s, "cr"] + add, 0.95) for s in ITEMS}
    b = bt if add == 0 else backtest_rule(q, COST_MONTHS, lift)
    runs[label] = b
    for period, mask in [("2026-06 to 2026-08", b["month"].isin(COST_MONTHS)), (STEADY, b["month"] == STEADY)]:
        g = compare(b[mask], "sku").reindex(ITEMS)
        g.loc["All"] = g.sum()
        g.loc["All", "short_rate_pct"] = g.loc["All", "rule_short"] / g.loc["All", "sold"] * 100
        rows.append(g.assign(variant=label, period=period, quantile=[q.get(s, np.nan) for s in g.index]))
rule_bt = pd.concat(rows).rename_axis("sku").reset_index()
rule_bt.to_csv(OUT / "prep_rule_backtest.csv", index=False)

high_short = compare(bt, "sku").query("short_rate_pct > 5").index.tolist()
print(f"Items with short rate > 5% at CR (Jun-Aug): {high_short}")
cols = ["variant", "quantile", "today_waste", "rule_waste", "rule_short", "short_rate_pct", "saving_per_month"]
steady = rule_bt[rule_bt["period"] == STEADY].set_index("sku")
print(f"{STEADY} (steady state)")
display(steady.loc[ITEMS, cols].round(2).sort_index(kind="stable"))

# Use the rule where it saves money in the steady month at CR+0.1 with promo days x lift (the plan).
plus = steady[steady["variant"] == "CR+0.1, promo x lift"]
RULE_ITEMS = [s for s in ITEMS if plus.loc[s, "saving_per_month"] > 0]
print(f"Rule items: {RULE_ITEMS} | saving at CR+0.1 in {STEADY}: "
      f"{plus.loc[RULE_ITEMS, 'saving_per_month'].sum() / 1000:.1f}k THB a month")

aug = bt[bt["month"] == STEADY]


def short_by_day_type(b):
    """Short rate of the rule items on normal and promo days, in the steady month."""
    a = b[(b["month"] == STEADY) & b["sku"].isin(RULE_ITEMS)]
    return a.groupby("promo_day").apply(lambda d: d["short"].sum() / d["sold"].sum() * 100, include_groups=False)


promo_short = pd.DataFrame({k: short_by_day_type(v) for k, v in runs.items()}).T
promo_short.columns = ["normal days", "promo days"]
w_n, s_n = normal_waste_short(aug["mu"], aug["sd"], aug["prep"])
check = pd.DataFrame({"rule_waste": aug.groupby("sku")["waste"].sum(), "normal_waste": pd.Series(w_n, aug.index).groupby(aug["sku"]).sum(),
                      "rule_short": aug.groupby("sku")["short"].sum(), "normal_short": pd.Series(s_n, aug.index).groupby(aug["sku"]).sum()}).reindex(ITEMS)
print(f"Short rate of rule items in {STEADY} (%):")
display(promo_short.round(1))
print("Normal-distribution check (used for Sep-Nov), cups in", STEADY)
display(check.round(0))

Items with short rate > 5% at CR (Jun-Aug): ['Guava', 'MixedBerryPremium', 'PassionFruit']
2026-08 (steady state)


,variant,quantile,today_waste,rule_waste,rule_short,short_rate_pct,saving_per_month
sku,,,,,,,
Guava,CR,0.44,409.0,403.0,722.0,10.13,-9272.09
Guava,CR+0.1,0.54,409.0,511.0,622.0,8.73,-9728.12
Guava,"CR+0.1, promo x lift",0.54,409.0,547.0,374.0,5.25,-7094.36
MixedBerryPremium,CR,0.05,632.0,113.0,550.0,21.67,26133.67
MixedBerryPremium,CR+0.1,0.15,632.0,207.0,415.0,16.35,21491.12
MixedBerryPremium,"CR+0.1, promo x lift",0.15,632.0,222.0,336.0,13.24,20897.59
PassionFruit,CR,0.16,552.0,131.0,936.0,16.35,8317.26
PassionFruit,CR+0.1,0.26,552.0,164.0,795.0,13.89,8088.92
PassionFruit,"CR+0.1, promo x lift",0.26,552.0,182.0,604.0,10.55,8678.98


Rule items: ['PassionFruit', 'MixedBerryPremium'] | saving at CR+0.1 in 2026-08: 29.6k THB a month
Short rate of rule items in 2026-08 (%):


,normal days,promo days
CR,14.0,36.7
CR+0.1,10.9,32.5
"CR+0.1, promo x lift",10.9,13.8


Normal-distribution check (used for Sep-Nov), cups in 2026-08


,rule_waste,normal_waste,rule_short,normal_short
sku,,,,
Watermelon,665.0,691.0,1109.0,777.0
Pineapple,625.0,555.0,598.0,589.0
Guava,403.0,351.0,722.0,497.0
PassionFruit,131.0,124.0,936.0,761.0
MixedBerryPremium,113.0,60.0,550.0,817.0


**Over Jun-Aug the rule costs more than today for every item but berry. Its 8-week window still holds the April peak in June: 15.3k wasted cups vs 3.0k today.**
August (steady state): the rule saves for berry (+20.9k a month at CR + 0.1, promo days x lift) and PassionFruit (+8.7k), and loses for the 3 cheap items (-7.1k to -10.0k), where today wastes only 3-5%.
The comparison favours today's way: today shows 0 short cups because sold-outs are not recorded, while every rule short cup counts as a lost sale.
Decision: rule for berry and PassionFruit at CR + 0.1 (safer: fewer customers turned away), +29.6k a month in August. Today's prep for the rest.
Promo days: at the normal-day level the rule items run 32.5% short on promo days (10.9% on normal days). Kitchens already scale up on promo days (section 1), so the plan preps the normal level x the measured lift: 13.8% short, same money.
Normal formula check: waste close to the rule; berry short higher (817 vs 550 cups), so Sep-Nov short numbers are on the safe side.

## Part B: forecast and 3-month outlook

## 4. Which forecast method wins? (train <= 31 May, test Jun-Aug)

In [8]:
TEST = pd.date_range("2026-06-01", "2026-08-31", freq="D")
fb = forecast_all(p, "2026-05-31", TEST).merge(p[DAY_KEYS + ["sold"]], on=DAY_KEYS, validate="many_to_one")
main = wape_bias(fb, "method")
by_sku = wape_bias(fb, ["method", "sku"])
monthly = wape_bias(fb.assign(month=fb["date"].dt.strftime("%Y-%m"))
                    .groupby(["method", "month", "kitchen", "sku"], as_index=False)[["sold", "forecast"]].sum(), "method")
winner = main["wape_pct"].idxmin()
display(main[["wape_pct", "bias_pct"]].round(1))
display(by_sku[["wape_pct", "bias_pct"]].unstack("method").reindex(ITEMS).round(1))
print(f"Winner: {winner} | monthly WAPE by method: {monthly['wape_pct'].round(1).to_dict()}")

# Same-season checks: one month ahead, no season change between train and test.
checks = {"Aug 2026 (train <= Jul 31)": ("2026-07-31", pd.date_range("2026-08-01", "2026-08-31"), p),
          "Oct 2025 (train <= Sep 30)": ("2025-09-30", pd.date_range("2025-10-01", "2025-10-31"), p[p["sku"] != BERRY])}
frames = [by_sku.reset_index().assign(test="Jun-Aug 2026 (train <= May 31)"),
          main.reset_index().assign(test="Jun-Aug 2026 (train <= May 31)", sku="All")]
for name, (end, dates, data) in checks.items():
    f = forecast_all(data, end, dates).merge(p[DAY_KEYS + ["sold"]], on=DAY_KEYS)
    frames.append(wape_bias(f, "method").reset_index().assign(test=name, sku="All"))
    print(name, wape_bias(f, "method")[["wape_pct", "bias_pct"]].round(1).to_dict("index"))
fcb = pd.concat(frames)[["test", "method", "sku", "sold", "forecast", "wape_pct", "bias_pct"]]
fcb["winner"] = fcb["method"] == winner
fcb.to_csv(OUT / "forecast_backtest.csv", index=False)

,wape_pct,bias_pct
method,,
holt_winters_damped,49.6,47.8
seasonal_naive,46.6,44.6
weekday_mean_8w,64.4,63.3


wape_pct                                 \
method            holt_winters_damped seasonal_naive weekday_mean_8w   
sku                                                                    
Watermelon                       47.2           44.0            61.8   
Pineapple                        46.6           42.4            61.6   
Guava                            55.3           51.5            62.5   
PassionFruit                     49.8           44.8            60.3   
MixedBerryPremium                60.5           70.0           104.3   

                             bias_pct                                 
method            holt_winters_damped seasonal_naive weekday_mean_8w  
sku                                                                   
Watermelon                       45.8           42.6            60.8  
Pineapple                        45.1           40.7            60.7  
Guava                            53.5           48.6            61.0  
PassionFruit                     47.9           41.7            58.8  
MixedBerryPremium                55.9           67.5           103.2

Winner: seasonal_naive | monthly WAPE by method: {'holt_winters_damped': 47.8, 'seasonal_naive': 44.6, 'weekday_mean_8w': 63.3}


Aug 2026 (train <= Jul 31) {'holt_winters_damped': {'wape_pct': 12.7, 'bias_pct': -2.1}, 'seasonal_naive': {'wape_pct': 15.5, 'bias_pct': -0.9}, 'weekday_mean_8w': {'wape_pct': 14.2, 'bias_pct': 3.2}}


Oct 2025 (train <= Sep 30) {'holt_winters_damped': {'wape_pct': 14.2, 'bias_pct': -2.4}, 'seasonal_naive': {'wape_pct': 14.2, 'bias_pct': -0.6}, 'weekday_mean_8w': {'wape_pct': 12.5, 'bias_pct': 0.6}}


In [9]:
hist = p[(p["date"] >= "2026-04-01") & (p["date"] <= "2026-08-31")].groupby("date")["sold"].sum()
fig = go.Figure()
fig.add_scatter(x=hist.index, y=hist, name="Actual", mode="lines", line=dict(color=INK, width=2))
for (name, g), color in zip(fb.groupby("method"), [UP, "#eb6834", "#1baf7a"]):
    s = g.groupby("date")["forecast"].sum()
    fig.add_scatter(x=s.index, y=s, name=name, mode="lines", line=dict(color=color, width=2))
fig.add_vline(x=pd.Timestamp("2026-05-31").timestamp() * 1000, line=dict(color="#c3c2b7", dash="dash", width=1))
fig.update_layout(
    title=subtitle(f"All 3 methods miss the rainy-season drop by {main['bias_pct'].min():.0f}-{main['bias_pct'].max():.0f}%; "
                   "none can see it without last year's Jun-Aug",
                   "Cups a day, all kitchens and items. Train up to 31 May (dashed line), test Jun-Aug 2026"),
    yaxis_title="Cups a day", height=460, legend=dict(orientation="h", y=1.02, x=1, xanchor="right"))
fig.show()

**Seasonal naive wins: WAPE 46.6%, bias +44.6%. All 3 methods are 45-63% too high.** Training ends in the Mar-May peak, and with no Jun-Aug last year no method sees the rainy-season drop.
Same-season checks (one month ahead): WAPE 12.5-15.5%, bias -2.4% to +3.2% for all 3. Sep follows a rainy August, so the method choice matters little there.

## 5. What will Sep-Nov 2026 sell?

In [10]:
fc = forecast_horizon(p, winner)
assert fc["forecast"].notna().all() and (fc["forecast"] >= 0).all()
assert len(fc) == (30 + 31 + 30) * 4 * 5, "every kitchen x item x day of Sep-Nov"

ly = p[p["month"].isin(["2025-09", "2025-10", "2025-11"])].copy()
ly["month"] = (ly["date"] + pd.DateOffset(years=1)).dt.strftime("%Y-%m")
fm = (fc.groupby(["month", "kitchen", "sku"]).agg(forecast=("forecast", "sum"), how=("how", "first"))
      .join(ly.groupby(["month", "kitchen", "sku"])["sold"].sum().rename("cups_2025")).reset_index())
fm["gap_vs_2025_pct"] = (fm["forecast"] / fm["cups_2025"] - 1) * 100
fm.to_csv(OUT / "forecast_monthly.csv", index=False)

item_m = fm.groupby(["sku", "month"])[["forecast", "cups_2025"]].sum()
item_m["gap_pct"] = (item_m["forecast"] / item_m["cups_2025"] - 1) * 100
display(item_m.unstack("month").reindex(ITEMS).round(1))
old = fm[fm["sku"] != BERRY]
big = old[old["gap_vs_2025_pct"].abs() > 15]
print(f"Old items vs Sep-Nov 2025: {old['forecast'].sum() / old['cups_2025'].sum() * 100 - 100:+.1f}% | "
      f"kitchen-item-months with a gap > 15%: {len(big)} of {len(old)}")

forecast                   cups_2025                    \
month              2026-09  2026-10  2026-11   2026-09  2026-10  2026-11   
sku                                                                        
Watermelon         16231.0  16304.8  19203.6   16513.0  16548.0  19265.0   
Pineapple          10580.0  10963.9  12278.0   10954.0  11334.0  12603.0   
Guava               6644.0   6932.7   7465.3    6853.0   7185.0   7690.0   
PassionFruit        5331.0   5478.8   6486.9    5420.0   5539.0   6514.0   
MixedBerryPremium   2456.2   2562.0   2473.0       0.0      0.0      0.0   

                  gap_pct                  
month             2026-09 2026-10 2026-11  
sku                                        
Watermelon           -1.7    -1.5    -0.3  
Pineapple            -3.4    -3.3    -2.6  
Guava                -3.0    -3.5    -2.9  
PassionFruit         -1.6    -1.1    -0.4  
MixedBerryPremium     inf     inf     inf

Old items vs Sep-Nov 2025: -2.0% | kitchen-item-months with a gap > 15%: 0 of 48


In [11]:
past = p.groupby("month")["sold"].sum()
fut = fc.groupby("month")["forecast"].sum()
fig = go.Figure()
fig.add_scatter(x=past.index, y=past / 1000, name="Actual", mode="lines+markers", line=dict(color=INK, width=2))
fig.add_scatter(x=[past.index[-1]] + list(fut.index), y=[past.iloc[-1] / 1000] + list(fut / 1000), name="Forecast",
                mode="lines+markers", line=dict(color=UP, width=2, dash="dash"))
fig.update_layout(
    title=subtitle(f"Sep-Nov 2026 forecast {fut.sum() / 1000:.0f}k cups, close to last year; "
                   "Nov is the only month that grows",
                   "Cups a month, all kitchens and items. Sep = seasonal naive, Oct-Nov = Sep x 2025 month ratio, berry flat"),
    yaxis_title="Cups (thousands)", height=440, legend=dict(orientation="h", y=1.02, x=1, xanchor="right"))
fig.show()

**Old items: -2.0% vs Sep-Nov 2025 (-0.3% to -3.5% per item and month); 0 of 48 kitchen-item-months more than 15% off.** Nov is +15-20% on Sep, as in 2025.
Berry flat at its August level (2,456-2,562 cups a month, ~82 a day). Risk: berry fell every month since launch, so flat is on the high side.

## 6. How much to prep and buy? (Action plan)

In [12]:
cv = demand_cv(p, "2026-08-31")
d = sc.build_inputs(fc, p, e, sheets, promo_inc, cv, COST_MONTHS)

KILL = {("RC101", "Watermelon"), ("RC101", "PassionFruit"), ("RC101", BERRY), ("RC103", "All")}
PRICES = {"Watermelon": 49, "PassionFruit": 69, BERRY: 99}
ACTION = dict(kill=KILL, prices=PRICES, rule_items=RULE_ITEMS)
zt = sc.rule_z(d, ACTION)
ACTION["z"] = zt["z"].to_dict()
SCEN = {
    "Base": {},
    "Action": ACTION,
    "Action + delist berry": {**ACTION, "delist": True},
    "Downside": {**ACTION, "demand_mult": 0.9, "fruit_mult": 1.15},
}
res = {name: sc.run(d, s) for name, s in SCEN.items()}
print("Inputs per item (Jun-Aug 2026 costs):")
display(d.groupby("sku")[["list_price", "comm_rate", "fruit_cup", "pack_cup", "co", "cv"]].first()
        .reindex(ITEMS).round(3))
print("CR at Action prices (rule items use CR + 0.1):")
display(zt.reindex(ITEMS).round(2))

Inputs per item (Jun-Aug 2026 costs):


,list_price,comm_rate,fruit_cup,pack_cup,co,cv
sku,,,,,,
Watermelon,45.0,0.275,7.945,11.0,13.951,0.096
Pineapple,49.0,0.269,9.249,11.0,15.228,0.120
Guava,49.0,0.279,10.236,11.5,16.238,0.178
PassionFruit,65.0,0.270,27.682,12.0,33.681,0.166
MixedBerryPremium,89.0,0.277,45.101,15.0,53.071,0.326


CR at Action prices (rule items use CR + 0.1):


,cu,co,cr,cr_used,z
sku,,,,,
Watermelon,16.25,13.95,0.54,0.64,0.35
Pineapple,14.81,15.23,0.49,0.59,0.24
Guava,13.19,16.24,0.45,0.55,0.12
PassionFruit,9.83,33.68,0.23,0.33,-0.45
MixedBerryPremium,10.33,53.07,0.16,0.26,-0.63


In [13]:
act = res["Action"].merge(d[["month", "kitchen", "sku", "fruit_cup"]], on=["month", "kitchen", "sku"])
inv = act[["month", "kitchen", "sku", "demand", "cups", "prep_cups", "waste_cups", "short_cups"]].copy()
inv["waste_rate_pct"] = inv["waste_cups"] / inv["prep_cups"] * 100
inv["fruit_budget_thb"] = inv["prep_cups"] * act["fruit_cup"]
inv["fruit_budget_downside_thb"] = inv["fruit_budget_thb"] * 1.15
inv["rule"] = np.where(inv["sku"].isin(RULE_ITEMS), "CR + 0.1 quantile", "today's prep")

# Cups to prep per day by weekday in Sep: forecast x Action demand change, then the rule or today's waste.
sep = fc[fc["month"] == "2026-09"].merge(
    (res["Action"].set_index(["month", "kitchen", "sku"])["demand"] / d.set_index(["month", "kitchen", "sku"])["demand"])
    .rename("factor").reset_index(), on=["month", "kitchen", "sku"])
sep = sep.merge(d[["month", "kitchen", "sku", "cv", "habit_waste_per_day"]], on=["month", "kitchen", "sku"])
sep["mu"] = sep["forecast"] * sep["factor"]
z = sep["sku"].map(ACTION["z"])
sep["prep_cups"] = np.where(sep["sku"].isin(RULE_ITEMS), sep["mu"] * (1 + z * sep["cv"]),
                            sep["mu"] + sep["habit_waste_per_day"])
wk = (sep.groupby(["kitchen", "sku", "dow"])[["mu", "prep_cups"]].mean().apply(np.ceil).reset_index()
      .rename(columns={"mu": "demand"}))
print("Sep 2026, cups to prep per day (Mon=0):")
display(wk.pivot_table(index=["kitchen", "sku"], columns="dow", values="prep_cups").astype(int))

inv_out = pd.concat([inv.assign(level="month"),
                     wk.assign(level="weekday (Sep 2026, per day)", month="2026-09",
                               rule=np.where(wk["sku"].isin(RULE_ITEMS), "CR + 0.1 quantile", "today's prep"))])
inv_out.to_csv(OUT / "inventory_plan.csv", index=False)
tot = inv.groupby("month")[["demand", "prep_cups", "waste_cups", "short_cups", "fruit_budget_thb",
                            "fruit_budget_downside_thb"]].sum()
display(tot.round(0))

Sep 2026, cups to prep per day (Mon=0):


dow                                  0    1    2    3    4    5    6
kitchen         sku                                                 
BKK_Ladprao     Guava               23   42   36   43   45   41   56
                MixedBerryPremium    8    7    7    9    9    9    9
                PassionFruit        19   20   27   22   20   18   22
                Pineapple           45   49   62   51   56   55   71
                Watermelon          75   74   69   82   69   71   88
BKK_Sukhumvit   Guava               57   61   62   55   48   56   55
                MixedBerryPremium   14   15   15   15   13   22   13
                PassionFruit        36   40   38   34   29   33   38
                Pineapple           90   80   85   86   86   98  109
                Watermelon         122  129  109  114  124  117  127
Pattaya_Beach   Guava               55   55   59   67   61   51   70
                MixedBerryPremium   15   16   17   16   16   25   23
                PassionFruit        39   37   40   53   43   53   64
                Pineapple           87   97   99  112   82  107  135
                Watermelon         143  110  121  119  131  136  170
Pattaya_Central Guava               67   64   94   51   85   75  104
                MixedBerryPremium   18   16   18   13   17   24   23
                PassionFruit        42   38   47   45   45   59   70
                Pineapple          104  121  111  112  109  131  143
                Watermelon         167  143  146  156  136  168  207

,demand,prep_cups,waste_cups,short_cups,fruit_budget_thb,fruit_budget_downside_thb
month,,,,,,
2026-09,37773.0,38691.0,1661.0,743.0,491394.0,565104.0
2026-10,38736.0,39683.0,1716.0,768.0,505912.0,581799.0
2026-11,44469.0,45320.0,1691.0,840.0,570609.0,656200.0


**Prep 38.7k / 39.7k / 45.3k cups in Sep / Oct / Nov. Fruit budget 491k / 506k / 571k THB (565k / 582k / 656k with fruit +15%).**
Berry and PassionFruit prep below demand: expected sold-out 17% and 8% of demand, waste 4.0% and 2.8% (from 20.5% and 8.8%).
The other 3 items keep today's prep: demand + 3-5 cups a day per kitchen. Weekday table = cups per day for Sep.

## 7. What happens to profit in each scenario? (Sep-Nov 2026)

In [14]:
summ = {n: sc.summary(r, sheets) for n, r in res.items()}
LINE_ORDER = ["gross_revenue", "commission", "fruit", "pack_labor", "waste_cost", "gross_profit", "overhead",
              "operating_profit", "gp_pct", "revenue_vs_budget", "gp_vs_budget", "cups", "short_cups"]
table = pd.concat({n: s[LINE_ORDER].T for n, s in summ.items()}, axis=1)
for n in SCEN:
    table[(n, "3 months")] = table[n].sum(axis=1)
    table.loc["gp_pct", (n, "3 months")] = summ[n]["gross_profit"].sum() / summ[n]["gross_revenue"].sum() * 100
show = table.copy()
money = [i for i in show.index if i not in ("gp_pct", "cups", "short_cups")]
show.loc[money] = (show.loc[money] / 1000).round(0)
show.loc[["cups", "short_cups"]] = (show.loc[["cups", "short_cups"]] / 1000).round(1)
show.loc["gp_pct"] = show.loc["gp_pct"].round(1)
print("THB thousands (cups in thousands)")
display(show)

# One model, so these must hold.
berry_gp = res["Action"].query("sku == @BERRY").groupby("month")["gross_profit"].sum()
assert np.allclose(summ["Action + delist berry"]["gross_profit"], summ["Action"]["gross_profit"] - berry_gp)
for n, s in summ.items():
    assert np.allclose(s["operating_profit"], s["gross_revenue"] - s["commission"] - s["fruit"] - s["pack_labor"]
                       - s["waste_cost"] - s["overhead"]), n
# Base per-cup revenue must match Jun-Aug actuals (promo mix and prices carried over correctly).
act_rpc = ja["gross_revenue"].sum() / ja["sold"].sum()
base_rpc = summ["Base"]["gross_revenue"].sum() / summ["Base"]["cups"].sum()
print(f"Revenue per cup: Base {base_rpc:.2f} vs Jun-Aug actual {act_rpc:.2f} THB ({base_rpc / act_rpc * 100 - 100:+.1f}%)")
assert abs(base_rpc / act_rpc - 1) < 0.03

THB thousands (cups in thousands)


Base                  Action                  \
month             2026-09 2026-10 2026-11 2026-09 2026-10 2026-11   
gross_revenue      2071.0  2127.0  2431.0  1954.0  2006.0  2299.0   
commission          565.0   580.0   663.0   533.0   548.0   628.0   
fruit               553.0   569.0   634.0   472.0   485.0   550.0   
pack_labor          472.0   484.0   547.0   422.0   433.0   496.0   
waste_cost           78.0    80.0    78.0    30.0    31.0    31.0   
gross_profit        403.0   413.0   509.0   497.0   510.0   595.0   
overhead            670.0   670.0   670.0   670.0   670.0   670.0   
operating_profit   -267.0  -257.0  -161.0  -173.0  -160.0   -75.0   
gp_pct               19.5    19.4    20.9    25.5    25.4    25.9   
revenue_vs_budget  -629.0  -573.0  -269.0  -746.0  -694.0  -401.0   
gp_vs_budget       -297.0  -287.0  -191.0  -203.0  -190.0  -105.0   
cups                 41.2    42.2    47.9    37.0    38.0    43.6   
short_cups            0.0     0.0     0.0     0.7     0.8     0.8   

                  Action + delist berry                 Downside          \
month                           2026-09 2026-10 2026-11  2026-09 2026-10   
gross_revenue                    1792.0  1837.0  2134.0   1758.0  1805.0   
commission                        489.0   502.0   583.0    480.0   493.0   
fruit                             396.0   407.0   473.0    488.0   502.0   
pack_labor                        397.0   406.0   470.0    380.0   389.0   
waste_cost                         26.0    27.0    27.0     32.0    33.0   
gross_profit                      484.0   495.0   581.0    379.0   388.0   
overhead                          670.0   670.0   670.0    670.0   670.0   
operating_profit                 -186.0  -175.0   -89.0   -291.0  -282.0   
gp_pct                             27.0    27.0    27.2     21.5    21.5   
revenue_vs_budget                -908.0  -863.0  -566.0   -942.0  -895.0   
gp_vs_budget                     -216.0  -205.0  -119.0   -321.0  -312.0   
cups                               35.4    36.2    41.9     33.3    34.2   
short_cups                          0.4     0.4     0.5      0.7     0.7   

                              Base   Action Action + delist berry Downside  
month             2026-11 3 months 3 months              3 months 3 months  
gross_revenue      2069.0   6629.0   6259.0                5764.0   5633.0  
commission          565.0   1809.0   1708.0                1574.0   1538.0  
fruit               569.0   1756.0   1507.0                1276.0   1560.0  
pack_labor          446.0   1503.0   1350.0                1273.0   1215.0  
waste_cost           33.0    236.0     92.0                  81.0     98.0  
gross_profit        456.0   1325.0   1602.0                1560.0   1223.0  
overhead            670.0   2010.0   2010.0                2010.0   2010.0  
operating_profit   -214.0   -685.0   -408.0                -450.0   -787.0  
gp_pct               22.0     20.0     25.6                  27.1     21.7  
revenue_vs_budget  -631.0  -1471.0  -1841.0               -2336.0  -2467.0  
gp_vs_budget       -244.0   -775.0   -498.0                -540.0   -877.0  
cups                 39.3    131.4    118.6                 113.5    106.8  
short_cups            0.8      0.0      2.4                   1.3      2.1

Revenue per cup: Base 50.45 vs Jun-Aug actual 50.74 THB (-0.6%)


In [15]:
# Lever bridge: add one lever at a time to the same model, so nothing is counted twice.
steps = [
    ("Base", {}),
    ("Kill RC101 W/PF/berry + RC103", {"kill": KILL}),
    ("Watermelon 49, PassionFruit 69", {"kill": KILL, "prices": {k: PRICES[k] for k in ["Watermelon", "PassionFruit"]}}),
    ("Berry 99", {"kill": KILL, "prices": PRICES}),
    ("Prep rule (berry, PassionFruit)", ACTION),
    ("Delist berry", {**ACTION, "delist": True}),
]
levers = {}
for eps in (-1.5, -0.5):
    op = {name: sc.summary(sc.run(d, s, eps=eps), sheets)["operating_profit"].sum() / 3 for name, s in steps}
    levers[eps] = pd.Series(op).diff().fillna(pd.Series(op))
    assert np.isclose(levers[eps].sum(), op["Delist berry"])
lever = levers[-1.5].rename("thb_per_month")
assert np.isclose(lever.loc[:"Prep rule (berry, PassionFruit)"].sum(), summ["Action"]["operating_profit"].sum() / 3)
lv = pd.DataFrame({"elasticity_-1.5": levers[-1.5], "elasticity_-0.5": levers[-0.5]}).rename_axis("step")
lv["kind"] = ["level"] + ["change"] * (len(lv) - 1)
lv.reset_index().to_csv(OUT / "scenario_levers.csv", index=False)
display((lv.drop(columns="kind") / 1000).round(1))

eps_low = sc.summary(sc.run(d, ACTION, eps=-0.5), sheets)["operating_profit"]
# Stress test at the cautious end of what we measured: per item, the lower of the day-level estimate and the CI low.
pr = pd.read_csv(OUT / "price_recommendation.csv").set_index("sku")
EPS_MEAS = pr[["day_elasticity", "ci_low"]].min(axis=1).dropna()
eps_meas = d["sku"].map(EPS_MEAS).fillna(-1.5)
act_meas = sc.summary(sc.run(d, ACTION, eps=eps_meas), sheets)["operating_profit"]
print("Cautious measured elasticity per item:", EPS_MEAS.round(2).to_dict())
print(f"Action op profit a month at the cautious measured elasticity: {act_meas.mean() / 1000:.1f}k "
      f"(vs Base {summ['Base']['operating_profit'].mean() / 1000:.1f}k)")
base_down = sc.summary(sc.run(d, {"demand_mult": 0.9, "fruit_mult": 1.15}), sheets)["operating_profit"]
print(f"Action op profit a month at elasticity -0.5: {eps_low.mean() / 1000:.1f}k vs -1.5: "
      f"{summ['Action']['operating_profit'].mean() / 1000:.1f}k | Base under the downside shock: {base_down.mean() / 1000:.1f}k")

# Berry with late-August fruit cost (last 4 weeks) instead of the Jun-Aug mean.
wf = sheets["weekly_fruit_cost"]
late = wf[(wf["sku"] == BERRY) & (wf["week_start"] >= "2026-08-03") & (wf["week_start"] <= "2026-08-24")]["fruit_cost_per_cup_thb"].mean()
d_late = d.copy()
d_late.loc[d_late["sku"] == BERRY, "fruit_cup"] = late
d_late.loc[d_late["sku"] == BERRY, "co"] += late - d.loc[d["sku"] == BERRY, "fruit_cup"].iloc[0]
berry_late = sc.run(d_late, ACTION).query("sku == @BERRY")["gross_profit"].sum() / 3
print(f"Berry GP-C a month in Action: {berry_gp.mean() / 1000:.1f}k at {d.loc[d['sku'] == BERRY, 'fruit_cup'].iloc[0]:.1f} THB fruit "
      f"(Jun-Aug mean) | {berry_late / 1000:.1f}k at {late:.1f} THB (Aug 3-30 weeks)")

,elasticity_-1.5,elasticity_-0.5
step,,
Base,-228.5,-228.5
Kill RC101 W/PF/berry + RC103,11.5,11.5
"Watermelon 49, PassionFruit 69",27.2,51.9
Berry 99,13.5,15.9
"Prep rule (berry, PassionFruit)",40.2,38.8
Delist berry,-13.9,-15.4


Cautious measured elasticity per item: {'Watermelon': -1.91, 'Pineapple': -2.12, 'Guava': -2.6, 'PassionFruit': -1.59, 'MixedBerryPremium': -1.51}
Action op profit a month at the cautious measured elasticity: -144.7k (vs Base -228.5k)


Action op profit a month at elasticity -0.5: -110.4k vs -1.5: -136.1k | Base under the downside shock: -368.6k


Berry GP-C a month in Action: 13.9k at 45.1 THB fruit (Jun-Aug mean) | 3.0k at 51.2 THB (Aug 3-30 weeks)


In [16]:
# Fruit cost check: same months of 2025 x year-on-year change (first 4 weeks of Sep 2025 -> last 4 of Aug 2026).
old_wf = wf[wf["sku"] != BERRY].assign(month=lambda x: x["week_start"].dt.strftime("%Y-%m"))
first = old_wf[old_wf["week_start"] < "2025-09-29"].groupby("sku")["fruit_cost_per_cup_thb"].mean()
last = old_wf[(old_wf["week_start"] >= "2026-08-03") & (old_wf["week_start"] <= "2026-08-24")].groupby("sku")["fruit_cost_per_cup_thb"].mean()
yoy = last / first
ly_cost = old_wf[old_wf["month"].isin(["2025-09", "2025-10", "2025-11"])].groupby(["sku", "month"])["fruit_cost_per_cup_thb"].mean()
alt = (ly_cost * yoy.reindex(ly_cost.index.get_level_values("sku")).to_numpy()).unstack()
alt.columns = ["2026-09", "2026-10", "2026-11"]
main_cost = d.groupby("sku")["fruit_cup"].first()
fruit_check = alt.assign(jun_aug_mean=main_cost.reindex(alt.index), yoy=yoy).reindex(ITEMS[:4])
display(fruit_check.round(2))
base = res["Base"].merge(d[["month", "kitchen", "sku"]], on=["month", "kitchen", "sku"])
base = base[base["sku"] != BERRY]
alt_long = alt.stack().rename("alt").reset_index().rename(columns={"level_1": "month"})
base = base.merge(alt_long, on=["sku", "month"])
diff = (base["cups"] * (base["sku"].map(main_cost) - base["alt"])).sum() / 3
print(f"With the 2025 x YoY fruit cost, Base GP-C changes by {diff / 1000:+.1f}k THB a month")

,2026-09,2026-10,2026-11,jun_aug_mean,yoy
sku,,,,,
Watermelon,7.92,7.97,7.99,7.94,1.00
Pineapple,9.82,9.53,9.84,9.25,0.97
Guava,9.95,9.97,9.42,10.24,1.07
PassionFruit,27.96,27.44,26.63,27.68,1.48


With the 2025 x YoY fruit cost, Base GP-C changes by -0.3k THB a month


In [17]:
rows = []
for n, r in res.items():
    rows.append(r.assign(scenario=n, kind="main"))
rows.append(sc.run(d, ACTION, eps=-0.5).assign(scenario="Action (elasticity -0.5)", kind="check"))
rows.append(sc.run(d, ACTION, eps=eps_meas).assign(scenario="Action (measured elasticity)", kind="check"))
rows.append(sc.run(d, {"demand_mult": 0.9, "fruit_mult": 1.15}).assign(scenario="Base downside", kind="check"))
pd.concat(rows).to_csv(OUT / "scenarios.csv", index=False)
pd.concat({n: s for n, s in summ.items()}, names=["scenario"]).reset_index().to_csv(OUT / "scenario_summary.csv", index=False)

fig = go.Figure()
for (n, s), color in zip(summ.items(), [TOTAL, UP, "#1baf7a", DOWN]):
    fig.add_bar(x=s.index, y=s["operating_profit"] / 1000, name=n, marker_color=color,
                hovertemplate=n + "<br>%{x}: %{y:,.0f}k THB<extra></extra>")
fig.add_hline(y=0, line=dict(color="#c3c2b7", width=1))
best = max(summ, key=lambda n: summ[n]["operating_profit"].sum())
fig.update_layout(
    barmode="group",
    title=subtitle(f"Every scenario loses money in Sep-Nov; {best} cuts the loss from "
                   f"{-summ['Base']['operating_profit'].sum() / 1000:.0f}k to {-summ[best]['operating_profit'].sum() / 1000:.0f}k",
                   "Operating profit (GP-C - 670k overhead) by month, THB thousands. Elasticity -1.5"),
    yaxis_title="THB thousands", height=460, legend=dict(orientation="h", y=1.02, x=1, xanchor="right"))
fig.show()

**Base: operating loss -685k in 3 months (-229k a month).** GP 19.5-20.9% on 2.1-2.4M revenue cannot cover 670k overhead.
**Action: -408k (+92k a month).** Prep rule +40k, Watermelon/PassionFruit prices +27k, berry 99 +14k, promo kills +12k. GP 25.6%, near the budget margin, but revenue is 6% below Base and 1.8M below budget.
Delist berry: -450k, 14k a month worse than berry at 99 with the rule (+13.9k a month). At late-August berry fruit (51.2 THB) berry still earns +3.0k, so keep it and let the stop rule decide.
Downside (Action + fruit +15% + demand -10%): -787k. Base under the same shock: -1,106k, so the plan still saves 106k a month in a bad case.
Checks: elasticity -0.5 gives -110k a month vs -136k at -1.5. Fruit 2025 x YoY changes Base GP by -0.3k a month, so the Jun-Aug mean holds.
No scenario reaches 0 in Sep-Nov: the gap left is overhead vs low-season volume (BKK kitchens, lesson 9).

## 8. Who does what, by when?

In [18]:
pr = pd.read_csv(OUT / "price_recommendation.csv").set_index("sku")


def cups_drop_at_breakeven(sku, new):
    """Cups drop (%) at which the price rise stops paying, from the lesson-6 break-even elasticity."""
    return (1 - (new / pr.loc[sku, "price_now"]) ** pr.loc[sku, "breakeven_elasticity"]) * 100


L = lever
# Sold-out share the plan expects, so the stop rule fires only when it is clearly worse than planned.
exp_short = (inv.groupby("sku")["short_cups"].sum() / inv.groupby("sku")["demand"].sum() * 100).round(0)
lift102 = LIFT[("RC102", "All")]
plan_rows = [
    ("Prep berry and PassionFruit at the CR + 0.1 quantile of the last 8 normal same-weekday days; "
     f"on promo days prep that level x the measured lift (RC102: {lift102:.2f})",
     "Kitchen managers (4), ops lead checks weekly", "2026-09-01",
     "Waste rate berry and PassionFruit; sold-out log (time and cups)",
     f"Plan expects sold-out on {exp_short[BERRY]:.0f}% (berry) and {exp_short['PassionFruit']:.0f}% (PassionFruit) "
     "of demand; if 5+ points above that for 2 weeks, move to CR + 0.2",
     L["Prep rule (berry, PassionFruit)"]),
    ("Raise Watermelon 45 -> 49 and PassionFruit 65 -> 69 THB on both apps",
     "Revenue manager", "2026-09-08",
     "Cups a day vs forecast; CM a day",
     f"Roll back if 4-week cups fall more than {cups_drop_at_breakeven('Watermelon', 49):.0f}% (Watermelon) or "
     f"{cups_drop_at_breakeven('PassionFruit', 69):.0f}% (PassionFruit) vs forecast",
     L["Watermelon 49, PassionFruit 69"]),
    ("Raise berry 89 -> 99 THB; decide delist on 2026-10-31",
     "Menu owner", "2026-09-01",
     "Berry GP-C per cup after waste, weekly",
     "Delist if GP-C per cup after waste is below 0 for 4 weeks in a row",
     L["Berry 99"]),
    ("Stop RC101 on Watermelon, PassionFruit, berry and RC103 on both apps",
     "Marketing / platform manager", "2026-09-01",
     "Item cups a day vs forecast; discount THB",
     "Restart a code only after a test shows net CM above 0",
     L["Kill RC101 W/PF/berry + RC103"]),
    ("4-week tests: Pineapple and Guava at 55 THB in one kitchen; RC101 Pineapple quiet hours, Guava 10%; "
     "RC102 at 12% without berry",
     "Revenue analyst", "2026-09-15",
     "Net CM vs a control kitchen or the same weekday baseline",
     "Stop a test if its net CM is below 0 after 4 weeks",
     0.0),
    ("Q4 review of BKK_Ladprao and BKK_Sukhumvit overhead (rent, staff hours)",
     "Finance / COO", "2026-11-01",
     "Operating profit per kitchen a month",
     "If Sep-Oct operating profit is still below 0, cost-cut plan or closure review by 2026-11-30",
     0.0),
]
plan = pd.DataFrame(plan_rows, columns=["action", "owner", "start_date", "kpi", "stop_rule", "thb_per_month"])
plan["thb_per_month"] = plan["thb_per_month"].round(-2)
plan["in_model"] = plan["thb_per_month"] != 0
plan.to_csv(OUT / "profit_plan.csv", index=False)
pd.set_option("display.max_colwidth", 70)
display(plan)
assert np.isclose(plan["thb_per_month"].sum(),
                  round(summ["Action"]["operating_profit"].sum() / 3 - summ["Base"]["operating_profit"].sum() / 3, -2), atol=300)

,action,owner,start_date,kpi,stop_rule,thb_per_month,in_model
0,Prep berry and PassionFruit at the CR + 0.1 quantile of the last 8...,"Kitchen managers (4), ops lead checks weekly",2026-09-01,Waste rate berry and PassionFruit; sold-out log (time and cups),Plan expects sold-out on 17% (berry) and 8% (PassionFruit) of dema...,40200.0,True
1,Raise Watermelon 45 -> 49 and PassionFruit 65 -> 69 THB on both apps,Revenue manager,2026-09-08,Cups a day vs forecast; CM a day,Roll back if 4-week cups fall more than 17% (Watermelon) or 19% (P...,27200.0,True
2,Raise berry 89 -> 99 THB; decide delist on 2026-10-31,Menu owner,2026-09-01,"Berry GP-C per cup after waste, weekly",Delist if GP-C per cup after waste is below 0 for 4 weeks in a row,13500.0,True
3,"Stop RC101 on Watermelon, PassionFruit, berry and RC103 on both apps",Marketing / platform manager,2026-09-01,Item cups a day vs forecast; discount THB,Restart a code only after a test shows net CM above 0,11500.0,True
4,4-week tests: Pineapple and Guava at 55 THB in one kitchen; RC101 ...,Revenue analyst,2026-09-15,Net CM vs a control kitchen or the same weekday baseline,Stop a test if its net CM is below 0 after 4 weeks,0.0,False
5,"Q4 review of BKK_Ladprao and BKK_Sukhumvit overhead (rent, staff h...",Finance / COO,2026-11-01,Operating profit per kitchen a month,"If Sep-Oct operating profit is still below 0, cost-cut plan or clo...",0.0,False


## Findings -> next steps
1. Berry is 39% of waste THB with 4.5% of cups; CR 0.05-0.16 for berry and PassionFruit -> prep below demand from 2026-09-01 with a sold-out log (+40k a month in the model; +29.6k in the August backtest).
2. The rule loses for the 3 cheap items (today wastes 3-5%, CR about 0.5) and needs 8 weeks after a season change; without the lift it runs 33% short on promo days -> keep today's prep for them; promo days = normal level x lift (14% short); review after 4 weeks of sold-out data.
3. Old items Sep-Nov 2026 within 3% of 2025 (Nov +15-20%), but a season change throws any method off by 45-63% -> re-forecast every month; berry flat forecast is on the high side.
4. Base loses 685k in Sep-Nov; Action 408k (+92k a month); Downside 787k vs 1,106k for Base -> run all 4 levers now; BKK overhead review on 2026-11-01 for the rest.
5. Berry at 99 + rule earns +3k to +14k a month, delisting is 3-14k a month worse -> keep berry, delist only if GP-C per cup is below 0 for 4 weeks in a row.